# Panel Dataset for Welfare State Capacity and Democratic Resilience (1990-2023)

## Overview

This notebook demonstrates the creation of a comprehensive panel dataset for mediated moderation analysis of welfare state capacity and democratic resilience.

**Research Question**: How does welfare state capacity mediate the relationship between inequality/educational polarization and democratic resilience in post-1990 democratizers?

**Dataset Features**:
- **Income inequality**: Gini coefficient (20-70 range)
- **Democratic resilience**: V-Dem Electoral Democracy Index (0-1) and Polity IV scores (-10 to +10)
- **Education**: Primary, secondary, tertiary enrollment rates (0-100%)
- **Welfare capacity**: Social spending as % of GDP (0-35%) and social protection coverage (%)
- **Derived indices**: Welfare capacity index and educational polarization index

**Coverage**: 782 country-year observations across 23 post-1990 democratizers (1990-2023)

**Data Sources**: Synthetic data generated based on realistic patterns from V-Dem, Polity IV, World Bank PIP, and OECD SOCX.

## Cell 1: Install Dependencies

This cell installs required packages. On Colab, core packages (numpy, pandas, etc.) are pre-installed. Locally, we install them at matching versions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2')

## Cell 2: Import Libraries

Import all required libraries for data processing, analysis, and visualization.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Cell 3: Data Loading Helper

This cell defines the data loading function that fetches data from GitHub (for Colab) with local fallback (for local testing).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-de87a0-welfare-state-capacity-as-a-conditioning/main/round-1/dataset-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """
    Load demo data from GitHub URL with local fallback.
    This pattern works both in Colab (loads from GitHub) and locally (loads from file).
    """
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print("✓ Loaded data from GitHub URL")
            return data
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            data = json.load(f)
            print("✓ Loaded data from local file")
            return data
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading function defined!")

## Cell 4: Load and Explore the Demo Data

Load the mini demo dataset and explore its structure.

In [ ]:
# Load the demo data
data = load_data()

# Explore the data structure
print("="*60)
print("DATA STRUCTURE")
print("="*60)

# Print metadata
if 'metadata' in data:
    print("\nMetadata:")
    for key, value in data['metadata'].items():
        if key != 'schema':  # Skip schema for brevity
            print(f"  {key}: {value}")

# Print dataset info
if 'datasets' in data:
    print("\nDatasets:")
    for dataset in data['datasets']:
        print(f"  Dataset: {dataset['dataset']}")
        print(f"  Number of examples: {len(dataset['examples'])}")
        
        # Show first example structure
        if len(dataset['examples']) > 0:
            print("\n  Example structure (first example):")
            example = dataset['examples'][0]
            print(f"    Input: {example['input'][:100]}...")
            print(f"    Output: {example['output']}")
            print(f"    Features: {example['metadata_feature_names']}")

## Cell 5: Convert JSON Data to DataFrame

Transform the JSON data into a pandas DataFrame for analysis. Parse the input JSON strings and extract relevant features.

In [ ]:
def convert_to_dataframe(data):
    """
    Convert the JSON dataset to a pandas DataFrame.
    Parses input JSON strings and extracts features and outputs.
    """
    rows = []
    
    for dataset in data['datasets']:
        for example in dataset['examples']:
            # Parse the input JSON string
            input_data = json.loads(example['input'])
            
            # Add the output (democratic resilience measure)
            input_data['vdem_polyarchy'] = float(example['output']) if example['output'] != 'null' else None
            
            rows.append(input_data)
    
    df = pd.DataFrame(rows)
    return df

# Convert to DataFrame
df = convert_to_dataframe(data)

print("="*60)
print("DATAFRAME CREATED")
print("="*60)
print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

## Cell 6: Configuration for Demo

Define configuration parameters for the demo. Since this is a data generation/processing script (not model training), the main parameters control data filtering and sampling.

In [ ]:
# Configuration parameters for the demo
# These control data filtering and which countries/years to include

CONFIG = {
    # Data filtering
    'demo_countries': ['ALB', 'ARM', 'BGR'],  # 3 countries for demo (minimum meaningful)
    'demo_years': list(range(1990, 2000)),     # 10 years for demo (1990-1999)
    'full_countries': 23,                      # Full dataset has 23 countries
    'full_years': list(range(1990, 2024)),     # Full dataset: 1990-2023
    
    # For scaling up in later cells
    'scale_up_countries': 5,   # Can increase to 5 countries for testing
    'scale_up_years': 15,      # Can increase to 15 years for testing
    
    # Original full dataset parameters (for reference)
    'original_num_countries': 23,
    'original_num_observations': 782,
    'original_years_range': (1990, 2023),
}

print("Configuration set!")
print(f"Demo will use {len(CONFIG['demo_countries'])} countries x {len(CONFIG['demo_years'])} years")
print(f"Full dataset has {CONFIG['original_num_countries']} countries, {CONFIG['original_num_observations']} observations")

## Cell 7: Identify Post-1990 Democratizers

This is Step 1 of the original script. Define the list of countries that democratized after 1990, based on Polity IV criteria (score <0 before 1990, >+5 after 1990).

In [ ]:
def identify_post_1990_democratizers():
    """
    Identify post-1990 democratizers using Polity IV criteria.
    Countries with Polity IV score <0 before 1990 and >+5 after 1990.
    """
    # Based on Polity IV data and literature on third-wave democratizers
    # This is a well-known list of countries that democratized after Cold War
    
    post_1990_democratizers = [
        # Eastern Europe and Former Soviet Union
        {"country": "Albania", "iso_code": "ALB", "transition_year": 1991},
        {"country": "Armenia", "iso_code": "ARM", "transition_year": 1991},
        {"country": "Azerbaijan", "iso_code": "AZE", "transition_year": 1991},
        {"country": "Belarus", "iso_code": "BLR", "transition_year": 1991},
        {"country": "Bulgaria", "iso_code": "BGR", "transition_year": 1990},
        {"country": "Croatia", "iso_code": "HRV", "transition_year": 2000},
        {"country": "Czech Republic", "iso_code": "CZE", "transition_year": 1990},
        {"country": "Estonia", "iso_code": "EST", "transition_year": 1991},
        {"country": "Georgia", "iso_code": "GEO", "transition_year": 1991},
        {"country": "Hungary", "iso_code": "HUN", "transition_year": 1990},
        {"country": "Latvia", "iso_code": "LVA", "transition_year": 1991},
        {"country": "Lithuania", "iso_code": "LTU", "transition_year": 1991},
        {"country": "Moldova", "iso_code": "MDA", "transition_year": 1991},
        {"country": "Poland", "iso_code": "POL", "transition_year": 1990},
        {"country": "Romania", "iso_code": "ROU", "transition_year": 1990},
        {"country": "Russia", "iso_code": "RUS", "transition_year": 1991},
        {"country": "Serbia", "iso_code": "SRB", "transition_year": 2000},
        {"country": "Slovakia", "iso_code": "SVK", "transition_year": 1990},
        {"country": "Slovenia", "iso_code": "SVN", "transition_year": 1991},
        {"country": "Ukraine", "iso_code": "UKR", "transition_year": 1991},
        
        # Latin America (continued democratization)
        {"country": "Argentina", "iso_code": "ARG", "transition_year": 1983},
        {"country": "Bolivia", "iso_code": "BOL", "transition_year": 1982},
        {"country": "Brazil", "iso_code": "BRA", "transition_year": 1985},
        {"country": "Chile", "iso_code": "CHL", "transition_year": 1990},
        {"country": "El Salvador", "iso_code": "SLV", "transition_year": 1992},
        {"country": "Guatemala", "iso_code": "GTM", "transition_year": 1996},
        {"country": "Honduras", "iso_code": "HND", "transition_year": 1982},
        {"country": "Mexico", "iso_code": "MEX", "transition_year": 2000},
        {"country": "Nicaragua", "iso_code": "NIC", "transition_year": 1990},
        {"country": "Paraguay", "iso_code": "PRY", "transition_year": 1989},
        {"country": "Peru", "iso_code": "PER", "transition_year": 2001},
        {"country": "Uruguay", "iso_code": "URY", "transition_year": 1985},
        
        # Africa
        {"country": "Benin", "iso_code": "BEN", "transition_year": 1991},
        {"country": "Botswana", "iso_code": "BWA", "transition_year": 1966},
        {"country": "Ghana", "iso_code": "GHA", "transition_year": 1992},
        {"country": "Kenya", "iso_code": "KEN", "transition_year": 2002},
        {"country": "Malawi", "iso_code": "MWI", "transition_year": 1994},
        {"country": "Mali", "iso_code": "MLI", "transition_year": 1992},
        {"country": "Namibia", "iso_code": "NAM", "transition_year": 1990},
        {"country": "Senegal", "iso_code": "SEN", "transition_year": 2000},
        {"country": "South Africa", "iso_code": "ZAF", "transition_year": 1994},
        {"country": "Tanzania", "iso_code": "TZA", "transition_year": 1995},
        {"country": "Uganda", "iso_code": "UGA", "transition_year": 1996},
        {"country": "Zambia", "iso_code": "ZMB", "transition_year": 1991},
        {"country": "Zimbabwe", "iso_code": "ZWE", "transition_year": 2009},
        
        # Asia
        {"country": "Indonesia", "iso_code": "IDN", "transition_year": 1998},
        {"country": "Mongolia", "iso_code": "MNG", "transition_year": 1990},
        {"country": "Philippines", "iso_code": "PHL", "transition_year": 1986},
        {"country": "South Korea", "iso_code": "KOR", "transition_year": 1987},
        {"country": "Taiwan", "iso_code": "TWN", "transition_year": 1996},
        {"country": "Thailand", "iso_code": "THA", "transition_year": 1992},
        
        # Southern Europe (earlier transitions)
        {"country": "Greece", "iso_code": "GRC", "transition_year": 1974},
        {"country": "Portugal", "iso_code": "PRT", "transition_year": 1974},
        {"country": "Spain", "iso_code": "ESP", "transition_year": 1975},
    ]
    
    return post_1990_democratizers

# Execute Step 1
print("="*60)
print("STEP 1: IDENTIFYING POST-1990 DEMOCRATIZERS")
print("="*60)

countries = identify_post_1990_democratizers()
print(f"\nFound {len(countries)} post-1990 democratizers")

# Show regional distribution
eastern_europe = ['ALB', 'ARM', 'AZE', 'BLR', 'BGR', 'HRV', 'CZE', 'EST', 'GEO', 'HUN', 'LVA', 'LTU', 'MDA', 'POL', 'ROU', 'RUS', 'SRB', 'SVK', 'SVN', 'UKR']
latin_america = ['ARG', 'BOL', 'BRA', 'CHL', 'SLV', 'GTM', 'HND', 'MEX', 'NIC', 'PRY', 'PER', 'URY']
africa = ['BEN', 'BWA', 'GHA', 'KEN', 'MWI', 'MLI', 'NAM', 'SEN', 'ZAF', 'TZA', 'UGA', 'ZMB', 'ZWE']
asia = ['IDN', 'MNG', 'PHL', 'KOR', 'TWN', 'THA']
southern_europe = ['GRC', 'PRT', 'ESP']

print("\nRegional distribution:")
print(f"  Eastern Europe/FSU: {len(eastern_europe)} countries")
print(f"  Latin America: {len(latin_america)} countries")
print(f"  Africa: {len(africa)} countries")
print(f"  Asia: {len(asia)} countries")
print(f"  Southern Europe: {len(southern_europe)} countries")

## Cell 8: Generate Synthetic Gini Coefficient Data

Step 2a: Create synthetic Gini coefficient (income inequality) data based on realistic regional patterns. In production, this would come from World Bank PIP or other sources.

In [ ]:
def create_synthetic_gini_data(countries: List[Dict], demo_mode=False, demo_countries=None) -> pd.DataFrame:
    """
    Create Gini coefficient data based on realistic patterns.
    In production, this would come from World Bank PIP or other sources.
    """
    print("Creating Gini coefficient data...")
    
    data = []
    years = list(range(1990, 2024))
    
    # Base Gini values by region (realistic ranges)
    base_gini = {
        "Eastern Europe": 30,
        "Latin America": 50,
        "Africa": 45,
        "Asia": 40,
        "Southern Europe": 35,
    }
    
    # Regional mapping
    eastern_europe = ['ALB', 'ARM', 'AZE', 'BLR', 'BGR', 'HRV', 'CZE', 'EST', 'GEO', 'HUN', 'LVA', 'LTU', 'MDA', 'POL', 'ROU', 'RUS', 'SRB', 'SVK', 'SVN', 'UKR']
    latin_america = ['ARG', 'BOL', 'BRA', 'CHL', 'SLV', 'GTM', 'HND', 'MEX', 'NIC', 'PRY', 'PER', 'URY']
    africa = ['BEN', 'BWA', 'GHA', 'KEN', 'MWI', 'MLI', 'NAM', 'SEN', 'ZAF', 'TZA', 'UGA', 'ZMB', 'ZWE']
    asia = ['IDN', 'MNG', 'PHL', 'KOR', 'TWN', 'THA']
    southern_europe = ['GRC', 'PRT', 'ESP']
    
    # Filter countries for demo mode
    if demo_mode and demo_countries:
        countries = [c for c in countries if c['iso_code'] in demo_countries]
        years = CONFIG['demo_years']  # Use shorter time range for demo
        print(f"  Demo mode: using {len(countries)} countries x {len(years)} years")
    
    for country in countries:
        iso = country['iso_code']
        
        # Determine region
        if iso in eastern_europe:
            region = "Eastern Europe"
        elif iso in latin_america:
            region = "Latin America"
        elif iso in africa:
            region = "Africa"
        elif iso in asia:
            region = "Asia"
        elif iso in southern_europe:
            region = "Southern Europe"
        else:
            region = "Other"
        
        base = base_gini.get(region, 40)
        
        # Generate Gini with realistic time trends and country-specific variation
        np.random.seed(hash(iso) % 2**32)  # Reproducible but country-specific
        
        for year in years:
            # Time trend: inequality tends to increase in transition economies
            years_since_transition = max(0, year - country['transition_year'])
            transition_effect = min(years_since_transition * 0.3, 10)  # Up to 10 points increase
            
            # Random variation
            noise = np.random.normal(0, 2)
            
            # Calculate Gini
            gini = base + transition_effect + noise
            gini = max(20, min(70, gini))  # Clip to realistic range
            
            data.append({
                'country_code': iso,
                'country': country['country'],
                'year': year,
                'gini': round(gini, 1),
                'region': region
            })
    
    df = pd.DataFrame(data)
    print(f"  Generated {len(df)} observations")
    return df

# Execute Step 2a - DEMO MODE (minimum parameters)
print("="*60)
print("STEP 2A: GENERATING GINI COEFFICIENT DATA (DEMO MODE)")
print("="*60)

gini_df = create_synthetic_gini_data(countries, demo_mode=True, demo_countries=CONFIG['demo_countries'])
print("\nFirst 10 rows of Gini data:")
print(gini_df.head(10))

## Cell 9: Generate Synthetic V-Dem Democracy Data

Step 2b: Create synthetic V-Dem Electoral Democracy Index data (v2x_polyarchy, 0-1 scale). This measures democratic resilience.

In [ ]:
def create_synthetic_vdem_data(countries: List[Dict], demo_mode=False, demo_countries=None) -> pd.DataFrame:
    """
    Create synthetic V-Dem Electoral Democracy Index data.
    v2x_polyarchy ranges from 0 to 1 (higher = more democratic).
    """
    print("Creating V-Dem Electoral Democracy Index data...")
    
    data = []
    years = list(range(1990, 2024))
    
    # Filter countries for demo mode
    if demo_mode and demo_countries:
        countries = [c for c in countries if c['iso_code'] in demo_countries]
        years = CONFIG['demo_years']
    
    for country in countries:
        iso = country['iso_code']
        transition_year = country['transition_year']
        
        np.random.seed(hash(iso + 'vdem') % 2**32)
        
        for year in years:
            if year < transition_year:
                # Pre-transition: low democracy scores
                base_score = 0.15 + np.random.normal(0, 0.05)
            else:
                # Post-transition: democracy consolidates over time
                years_since = year - transition_year
                # Logistic-like improvement
                if years_since <= 0:
                    base_score = 0.2
                elif years_since <= 5:
                    base_score = 0.2 + (0.5 * years_since / 5)
                elif years_since <= 15:
                    base_score = 0.7 + (0.2 * (years_since - 5) / 10)
                else:
                    base_score = 0.85 + np.random.normal(0, 0.05)
                
                base_score += np.random.normal(0, 0.03)  # Add noise
            
            # Ensure valid range [0, 1]
            vdem_score = max(0.0, min(1.0, base_score))
            
            data.append({
                'country_code': iso,
                'country': country['country'],
                'year': year,
                'vdem_polyarchy': round(vdem_score, 3)
            })
    
    df = pd.DataFrame(data)
    print(f"  Generated {len(df)} observations")
    return df

# Execute Step 2b - DEMO MODE
print("="*60)
print("STEP 2B: GENERATING V-DEM DEMOCRACY DATA (DEMO MODE)")
print("="*60)

vdem_df = create_synthetic_vdem_data(countries, demo_mode=True, demo_countries=CONFIG['demo_countries'])
print("\nFirst 10 rows of V-Dem data:")
print(vdem_df.head(10))

## Cell 10: Generate Additional Synthetic Datasets

Steps 2c-2e: Generate Polity IV scores, education enrollment rates, and welfare state capacity data. For the demo, we'll create simplified versions.

In [ ]:
def create_synthetic_polity_data(countries: List[Dict], demo_mode=False, demo_countries=None) -> pd.DataFrame:
    """Create synthetic Polity IV scores (-10 to +10)"""
    print("Creating Polity IV data...")
    data = []
    years = list(range(1990, 2024)) if not demo_mode else CONFIG['demo_years']
    
    if demo_mode and demo_countries:
        countries = [c for c in countries if c['iso_code'] in demo_countries]
    
    for country in countries:
        iso = country['iso_code']
        transition_year = country['transition_year']
        
        np.random.seed(hash(iso + 'polity') % 2**32)
        
        for year in years:
            if year < transition_year:
                polity_score = -5 + np.random.normal(0, 2)
            else:
                years_since = year - transition_year
                if years_since <= 10:
                    polity_score = -5 + (15 * years_since / 10) + np.random.normal(0, 1)
                else:
                    polity_score = 8 + np.random.normal(0, 1.5)
            
            polity_score = max(-10, min(10, polity_score))
            
            data.append({
                'country_code': iso,
                'country': country['country'],
                'year': year,
                'polity_iv': round(polity_score, 1)
            })
    
    df = pd.DataFrame(data)
    print(f"  Generated {len(df)} observations")
    return df

def create_synthetic_education_data(countries: List[Dict], demo_mode=False, demo_countries=None) -> pd.DataFrame:
    """Create synthetic education enrollment rates (0-100%)"""
    print("Creating education enrollment data...")
    data = []
    years = list(range(1990, 2024)) if not demo_mode else CONFIG['demo_years']
    
    if demo_mode and demo_countries:
        countries = [c for c in countries if c['iso_code'] in demo_countries]
    
    for country in countries:
        iso = country['iso_code']
        
        np.random.seed(hash(iso + 'edu') % 2**32)
        
        base_primary = 90 + np.random.normal(0, 5)
        base_secondary = 70 + np.random.normal(0, 10)
        base_tertiary = 30 + np.random.normal(0, 10)
        
        for year in years:
            # Education improves over time
            year_effect = (year - 1990) * 0.5
            
            primary = max(50, min(100, base_primary + year_effect + np.random.normal(0, 2)))
            secondary = max(20, min(100, base_secondary + year_effect + np.random.normal(0, 3)))
            tertiary = max(5, min(80, base_tertiary + year_effect * 0.3 + np.random.normal(0, 2)))
            
            data.append({
                'country_code': iso,
                'country': country['country'],
                'year': year,
                'primary_enrollment': round(primary, 1),
                'secondary_enrollment': round(secondary, 1),
                'tertiary_enrollment': round(tertiary, 1)
            })
    
    df = pd.DataFrame(data)
    print(f"  Generated {len(df)} observations")
    return df

def create_synthetic_welfare_data(countries: List[Dict], demo_mode=False, demo_countries=None) -> pd.DataFrame:
    """Create synthetic welfare state capacity data"""
    print("Creating welfare state capacity data...")
    data = []
    years = list(range(1990, 2024)) if not demo_mode else CONFIG['demo_years']
    
    if demo_mode and demo_countries:
        countries = [c for c in countries if c['iso_code'] in demo_countries]
    
    for country in countries:
        iso = country['iso_code']
        
        np.random.seed(hash(iso + 'welfare') % 2**32)
        
        base_spending = 15 + np.random.normal(0, 5)  # Social spending as % GDP
        
        for year in years:
            # Welfare state grows over time
            year_effect = (year - 1990) * 0.2
            
            spending = max(5, min(35, base_spending + year_effect + np.random.normal(0, 1)))
            coverage = max(20, min(100, spending * 3 + np.random.normal(0, 5)))
            
            data.append({
                'country_code': iso,
                'country': country['country'],
                'year': year,
                'social_spending_gdp': round(spending, 1),
                'social_protection_coverage': round(coverage, 1)
            })
    
    df = pd.DataFrame(data)
    print(f"  Generated {len(df)} observations")
    return df

# Execute Steps 2c-2e - DEMO MODE
print("="*60)
print("STEPS 2C-2E: GENERATING ADDITIONAL DATASETS (DEMO MODE)")
print("="*60)

polity_df = create_synthetic_polity_data(countries, demo_mode=True, demo_countries=CONFIG['demo_countries'])
education_df = create_synthetic_education_data(countries, demo_mode=True, demo_countries=CONFIG['demo_countries'])
welfare_df = create_synthetic_welfare_data(countries, demo_mode=True, demo_countries=CONFIG['demo_countries'])

print("\n✓ All synthetic datasets generated successfully!")

## Cell 11: Merge All Datasets

Step 3: Merge all individual datasets into a unified panel dataset using country_code and year as keys.

In [ ]:
def merge_datasets(datasets: Dict, countries: List[Dict], demo_mode=False) -> pd.DataFrame:
    """
    Merge all individual datasets into a unified panel dataset.
    Uses country_code and year as merge keys.
    """
    print("Merging datasets...")
    
    # Start with the first dataset
    merged = datasets['gini'].copy()
    print(f"  Starting with Gini data: {len(merged)} rows")
    
    # Merge V-Dem data
    if 'vdem' in datasets:
        merged = pd.merge(merged, datasets['vdem'], on=['country_code', 'country', 'year'], how='left')
        print(f"  After merging V-Dem: {len(merged)} rows")
    
    # Merge Polity IV data
    if 'polity' in datasets:
        merged = pd.merge(merged, datasets['polity'], on=['country_code', 'country', 'year'], how='left')
        print(f"  After merging Polity IV: {len(merged)} rows")
    
    # Merge education data
    if 'education' in datasets:
        merged = pd.merge(merged, datasets['education'], on=['country_code', 'country', 'year'], how='left')
        print(f"  After merging education: {len(merged)} rows")
    
    # Merge welfare data
    if 'welfare' in datasets:
        merged = pd.merge(merged, datasets['welfare'], on=['country_code', 'country', 'year'], how='left')
        print(f"  After merging welfare: {len(merged)} rows")
    
    # Add post-1990 democratizer flag and transition year
    country_info = {c['iso_code']: c for c in countries}
    
    merged['post_1990_democratizer'] = merged['country_code'].apply(
        lambda x: 1 if x in [c['iso_code'] for c in countries if c['transition_year'] >= 1990] else 0
    )
    
    merged['transition_year'] = merged['country_code'].apply(
        lambda x: country_info[x]['transition_year'] if x in country_info else None
    )
    
    # Calculate derived indices (simplified for demo)
    # Welfare capacity index: combination of social spending and coverage
    if 'social_spending_gdp' in merged.columns and 'social_protection_coverage' in merged.columns:
        merged['welfare_capacity_index'] = (
            (merged['social_spending_gdp'] / 35 * 0.5) + 
            (merged['social_protection_coverage'] / 100 * 0.5)
        )
        merged['welfare_capacity_index'] = merged['welfare_capacity_index'].round(3)
    
    # Educational polarization index: inverse of enrollment (simplified)
    if 'tertiary_enrollment' in merged.columns:
        merged['educational_polarization'] = (100 - merged['tertiary_enrollment']) / 100
        merged['educational_polarization'] = merged['educational_polarization'].round(3)
    
    print(f"\n  Final merged dataset: {len(merged)} rows, {len(merged.columns)} columns")
    
    return merged

# Execute Step 3 - Merge datasets
print("="*60)
print("STEP 3: MERGING DATASETS")
print("="*60)

# Collect all datasets into a dictionary
datasets = {
    'gini': gini_df,
    'vdem': vdem_df,
    'polity': polity_df,
    'education': education_df,
    'welfare': welfare_df
}

merged_df = merge_datasets(datasets, countries, demo_mode=True)

print("\n" + "="*60)
print("MERGED DATASET PREVIEW")
print("="*60)
print(f"\nShape: {merged_df.shape}")
print(f"\nColumns: {list(merged_df.columns)}")
print(f"\nFirst 5 rows:")
print(merged_df.head())

## Cell 12: Validate the Dataset

Step 4: Validate the merged dataset - check correlations, ranges, missing data, and panel balance.

In [ ]:
def validate_dataset(df: pd.DataFrame) -> Dict:
    """
    Validate the merged dataset.
    Checks correlations, ranges, missing data, and panel balance.
    """
    print("Validating merged dataset...")
    
    validation = {
        'total_observations': int(len(df)),
        'countries': int(df['country_code'].nunique()),
        'years_covered': f"{df['year'].min()}-{df['year'].max()}",
        'missing_data': {},
        'checks': []
    }
    
    # Check missing data
    for col in df.columns:
        if col not in ['country_code', 'country', 'region']:
            missing_pct = (df[col].isna().sum() / len(df) * 100)
            validation['missing_data'][col] = round(missing_pct, 2)
    
    # Validation checks
    
    # Check 1: V-Dem and Polity IV correlation > 0.7
    if 'vdem_polyarchy' in df.columns and 'polity_iv' in df.columns:
        valid_mask = df['vdem_polyarchy'].notna() & df['polity_iv'].notna()
        if valid_mask.sum() > 10:
            correlation = df.loc[valid_mask, 'vdem_polyarchy'].corr(df.loc[valid_mask, 'polity_iv'])
            validation['checks'].append(('V-Dem/Polity IV correlation > 0.7', round(correlation, 3), '>0.7', correlation > 0.7))
    
    # Check 2: Gini coefficient in plausible range (20-70)
    if 'gini' in df.columns:
        gini_valid = df['gini'].between(20, 70).mean() * 100
        validation['checks'].append(('Gini in range [20-70]', round(gini_valid, 1), '>95%', gini_valid > 95))
    
    # Check 3: V-Dem in range [0-1]
    if 'vdem_polyarchy' in df.columns:
        vdem_valid = df['vdem_polyarchy'].between(0, 1).mean() * 100
        validation['checks'].append(('V-Dem in range [0-1]', round(vdem_valid, 1), '100%', vdem_valid == 100))
    
    # Check 4: Polity IV in range [-10, +10]
    if 'polity_iv' in df.columns:
        polity_valid = df['polity_iv'].between(-10, 10).mean() * 100
        validation['checks'].append(('Polity IV in range [-10, +10]', round(polity_valid, 1), '100%', polity_valid == 100))
    
    # Check 5: No duplicate observations
    duplicates = df.duplicated(subset=['country_code', 'year']).sum()
    validation['checks'].append(('No duplicate observations', int(duplicates), '0', duplicates == 0))
    
    # Check 6: Panel balance (all countries have same number of years)
    years_per_country = df.groupby('country_code')['year'].count()
    panel_balanced = years_per_country.nunique() == 1
    validation['checks'].append(('Panel balance (all countries same years)', int(years_per_country.iloc[0]), 'Expected years', panel_balanced))
    
    return validation

# Execute Step 4 - Validate dataset
print("="*60)
print("STEP 4: VALIDATING MERGED DATASET")
print("="*60)

validation = validate_dataset(merged_df)

print(f"\nTotal observations: {validation['total_observations']}")
print(f"Countries: {validation['countries']}")
print(f"Years covered: {validation['years_covered']}")

print("\nMissing data:")
for col, pct in validation['missing_data'].items():
    print(f"  {col}: {pct}%")

print("\nValidation checks:")
for check_name, actual, expected, passed in validation['checks']:
    status = "✓" if passed else "✗"
    print(f"  {status} {check_name}: {actual} (expected {expected})")

## Cell 13: Visualize Key Relationships

Create visualizations to explore the relationships between inequality, welfare capacity, and democratic resilience.

In [ ]:
# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(16, 12))

# Plot 1: Gini coefficient over time by country
ax1 = plt.subplot(2, 3, 1)
for country in merged_df['country_code'].unique()[:3]:  # First 3 countries
    country_data = merged_df[merged_df['country_code'] == country]
    ax1.plot(country_data['year'], country_data['gini'], marker='o', label=country, markersize=4)
ax1.set_xlabel('Year')
ax1.set_ylabel('Gini Coefficient')
ax1.set_title('Income Inequality Over Time (Demo Countries)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: V-Dem democracy index over time
ax2 = plt.subplot(2, 3, 2)
for country in merged_df['country_code'].unique()[:3]:
    country_data = merged_df[merged_df['country_code'] == country]
    ax2.plot(country_data['year'], country_data['vdem_polyarchy'], marker='s', label=country, markersize=4)
ax2.set_xlabel('Year')
ax2.set_ylabel('V-Dem Polyarchy Index')
ax2.set_title('Democratic Resilience Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Scatter plot - Gini vs V-Dem
ax3 = plt.subplot(2, 3, 3)
valid_data = merged_df[merged_df['vdem_polyarchy'].notna() & merged_df['gini'].notna()]
ax3.scatter(valid_data['gini'], valid_data['vdem_polyarchy'], alpha=0.6, s=50)
ax3.set_xlabel('Gini Coefficient')
ax3.set_ylabel('V-Dem Polyarchy Index')
ax3.set_title('Inequality vs Democratic Resilience')
ax3.grid(True, alpha=0.3)

# Add correlation coefficient
if len(valid_data) > 2:
    corr = valid_data['gini'].corr(valid_data['vdem_polyarchy'])
    ax3.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax3.transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Welfare capacity index over time
ax4 = plt.subplot(2, 3, 4)
if 'welfare_capacity_index' in merged_df.columns:
    for country in merged_df['country_code'].unique()[:3]:
        country_data = merged_df[merged_df['country_code'] == country]
        ax4.plot(country_data['year'], country_data['welfare_capacity_index'], marker='^', label=country, markersize=4)
    ax4.set_xlabel('Year')
    ax4.set_ylabel('Welfare Capacity Index')
    ax4.set_title('Welfare State Capacity Over Time')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

# Plot 5: Social spending vs V-Dem
ax5 = plt.subplot(2, 3, 5)
if 'social_spending_gdp' in merged_df.columns:
    valid_data = merged_df[merged_df['vdem_polyarchy'].notna() & merged_df['social_spending_gdp'].notna()]
    ax5.scatter(valid_data['social_spending_gdp'], valid_data['vdem_polyarchy'], alpha=0.6, s=50, c='green')
    ax5.set_xlabel('Social Spending (% GDP)')
    ax5.set_ylabel('V-Dem Polyarchy Index')
    ax5.set_title('Welfare Spending vs Democratic Resilience')
    ax5.grid(True, alpha=0.3)
    
    if len(valid_data) > 2:
        corr = valid_data['social_spending_gdp'].corr(valid_data['vdem_polyarchy'])
        ax5.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax5.transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 6: Education enrollment over time
ax6 = plt.subplot(2, 3, 6)
if 'tertiary_enrollment' in merged_df.columns:
    for country in merged_df['country_code'].unique()[:3]:
        country_data = merged_df[merged_df['country_code'] == country]
        ax6.plot(country_data['year'], country_data['tertiary_enrollment'], marker='d', label=country, markersize=4)
    ax6.set_xlabel('Year')
    ax6.set_ylabel('Tertiary Enrollment (%)')
    ax6.set_title('Tertiary Education Over Time')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Panel Dataset Exploration: Welfare State Capacity & Democratic Resilience', fontsize=14, y=1.02)
plt.show()

print("✓ Visualizations created successfully!")

## Cell 14: Summary Statistics and Key Findings

Display summary statistics and highlight key findings from the dataset.

In [ ]:
print("="*60)
print("SUMMARY STATISTICS AND KEY FINDINGS")
print("="*60)

# Summary statistics for key variables
print("\n📊 SUMMARY STATISTICS (Demo Dataset)")
print("-" * 60)

numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
summary = merged_df[numeric_cols].describe()
print(summary.round(2))

# Key correlations
print("\n🔗 KEY CORRELATIONS")
print("-" * 60)

correlations = []
if 'gini' in merged_df.columns and 'vdem_polyarchy' in merged_df.columns:
    corr = merged_df['gini'].corr(merged_df['vdem_polyarchy'])
    correlations.append(('Gini ↔ V-Dem (Inequality ↔ Democracy)', corr))

if 'social_spending_gdp' in merged_df.columns and 'vdem_polyarchy' in merged_df.columns:
    corr = merged_df['social_spending_gdp'].corr(merged_df['vdem_polyarchy'])
    correlations.append(('Social Spending ↔ V-Dem (Welfare ↔ Democracy)', corr))

if 'gini' in merged_df.columns and 'social_spending_gdp' in merged_df.columns:
    corr = merged_df['gini'].corr(merged_df['social_spending_gdp'])
    correlations.append(('Gini ↔ Social Spending (Inequality ↔ Welfare)', corr))

if 'welfare_capacity_index' in merged_df.columns and 'vdem_polyarchy' in merged_df.columns:
    corr = merged_df['welfare_capacity_index'].corr(merged_df['vdem_polyarchy'])
    correlations.append(('Welfare Capacity ↔ V-Dem (Capacity ↔ Democracy)', corr))

for name, corr in correlations:
    print(f"  {name}: r = {corr:.3f}")

# Regional comparison (if region column exists)
if 'region' in merged_df.columns:
    print("\n🌍 REGIONAL COMPARISON (Average V-Dem by Region)")
    print("-" * 60)
    regional_avg = merged_df.groupby('region')['vdem_polyarchy'].mean().sort_values(ascending=False)
    for region, avg in regional_avg.items():
        print(f"  {region}: {avg:.3f}")

print("\n✓ Demo dataset exploration complete!")
print("\n" + "="*60)
print("NOTE: This demo uses synthetic data based on realistic patterns.")
print("      Full dataset has 782 observations across 23 countries.")
print("="*60)

## Cell 15: Export Demo Dataset to JSON

Convert the demo DataFrame back to the JSON format expected by the aii-json schema (full/mini/preview structure).

In [ ]:
def export_demo_dataset(df: pd.DataFrame, validation: Dict, output_filename="demo_data_out.json"):
    """
    Export the demo dataset to JSON in the aii-json compatible format.
    """
    print("Exporting demo dataset to JSON...")
    
    # Define schema
    schema = {
        "country_code": "ISO 3-letter country code",
        "country_name": "Country name",
        "year": "Year (1990-2023)",
        "gini": "Gini coefficient (0-100 scale)",
        "vdem_polyarchy": "V-Dem Electoral Democracy Index (0-1)",
        "polity_iv": "Polity IV score (-10 to +10)",
        "primary_enrollment": "Primary enrollment rate (%)",
        "secondary_enrollment": "Secondary enrollment rate (%)",
        "tertiary_enrollment": "Tertiary enrollment rate (%)",
        "social_spending_gdp": "Social spending as % of GDP",
        "social_protection_coverage": "Social protection coverage (%)",
        "welfare_capacity_index": "Welfare state capacity index (0-1)",
        "educational_polarization": "Educational polarization index (0-1)",
        "post_1990_democratizer": "Flag: post-1990 democratizer (1=yes, 0=no)",
        "transition_year": "Year of democratic transition"
    }
    
    # Prepare export data structure
    export_data = {
        "metadata": {
            "title": "Panel Dataset for Welfare State Capacity and Democratic Resilience (1990-2023) - DEMO",
            "description": "Demo version of unified panel dataset (subset of data)",
            "source": "Synthetic data based on patterns from V-Dem, Polity IV, World Bank PIP, OECD SOCX",
            "years": f"{df['year'].min()}-{df['year'].max()}",
            "countries": int(df['country_code'].nunique()),
            "observations": int(len(df)),
            "schema": schema,
            "validation": validation
        },
        "datasets": [
            {
                "dataset": "welfare_democracy_panel",
                "examples": []
            }
        ]
    }
    
    # Convert each row to example format
    for _, row in df.iterrows():
        example = {
            "input": json.dumps({
                "country_code": row["country_code"],
                "country_name": row["country"],
                "year": int(row["year"]),
                "gini": float(row["gini"]) if pd.notna(row["gini"]) else None,
                "vdem_polyarchy": float(row["vdem_polyarchy"]) if pd.notna(row["vdem_polyarchy"]) else None,
                "polity_iv": float(row["polity_iv"]) if pd.notna(row["polity_iv"]) else None,
            }),
            "output": str(float(row["vdem_polyarchy"])) if pd.notna(row["vdem_polyarchy"]) else "null",
            "metadata_fold": None,
            "metadata_feature_names": list(schema.keys())
        }
        export_data["datasets"][0]["examples"].append(example)
    
    # Save to file
    with open(output_filename, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)
    
    print(f"  Demo dataset exported to: {output_filename}")
    print(f"  Total examples: {len(export_data['datasets'][0]['examples'])}")
    
    return output_filename

# Execute export
print("="*60)
print("EXPORTING DEMO DATASET")
print("="*60)

exported_file = export_demo_dataset(merged_df, validation)
print(f"\n✓ Dataset exported successfully!")